# OneVoice V2 — English construction ASR
Chỉ chạy sau khi đã sinh audio English V2.1 và manifest qua audit tối thiểu sáu speaker/voice.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, subprocess, sys
GITHUB_REPO = 'https://github.com/Platypus27-coder/OneVoice.git'
BRANCH = 'main'
REPO = Path('/content/OneVoice')
MYDRIVE = Path('/content/drive/MyDrive')
WORK_ROOT = MYDRIVE / 'OneVoice'
if (REPO / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, GITHUB_REPO, str(REPO)], check=True)
os.environ['MODELSCOPE_CACHE'] = str(WORK_ROOT / 'model_cache/modelscope')
os.chdir(REPO)
os.environ['PYTHONUNBUFFERED'] = '1'
# funasr_onnx exports a missing SenseVoice ONNX bundle on first use; funasr, onnx and onnxscript are required for that export.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy', 'PyYAML', 'soundfile', 'librosa', 'funasr', 'funasr_onnx', 'modelscope', 'onnx', 'onnxscript'], check=True)
MANIFEST = MYDRIVE / 'onevoice_audio_v2_1/manifest.jsonl'
if not MANIFEST.is_file():
    raise FileNotFoundError('English V2.1 audio has not been generated yet; run this notebook only after its manifest exists')
REPORT_ROOT = WORK_ROOT / 'reports/en_asr'

def run_streaming(command, label):
    print(f'\n[{label}] > ' + ' '.join(map(str, command)), flush=True)
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='', flush=True)
    code = process.wait()
    print(f'[{label}] exit code: {code}', flush=True)
    if code:
        raise RuntimeError(f'{label} failed; the complete subprocess log is printed above.')

print('Source:', REPO, '| Data:', MANIFEST, '| Reports:', REPORT_ROOT)


In [ ]:
import json, shutil
audit_dir = REPORT_ROOT / 'audit'
audit_file = audit_dir / 'audit.json'
def audit_passed(path):
    try:
        return path.is_file() and json.loads(path.read_text(encoding='utf-8')).get('passed') is True
    except (OSError, json.JSONDecodeError):
        return False

if not audit_passed(audit_file):
    had_physical_audit = audit_file.is_file()
    if had_physical_audit:
        shutil.copy2(audit_file, audit_dir / 'physical_before_metadata_reconcile.json')
    run_streaming([sys.executable, 'scripts/reconcile_manifest_splits.py', str(MANIFEST), '--language', 'en', '--in-place', '--report', str(audit_dir / 'split_reconcile.json')], 'Reconcile English manifest splits')
    command = [sys.executable, 'scripts/audit_audio_dataset.py', str(MANIFEST), '--language', 'en', '--min-speakers', '6', '--require-realized-snr', '--expected-clean', '8064', '--expected-noisy', '16128', '--report-dir', str(audit_dir)]
    if had_physical_audit:
        command.append('--logical-only')
    run_streaming(command, 'EN-ASR dataset audit')
else:
    print('EN-ASR dataset audit already complete on Drive; skipping.', flush=True)

for audio in ('clean', 'noisy'):
    report_dir = REPORT_ROOT / audio
    required = ('aggregate.json', 'predictions.csv', 'run_manifest.json')
    if all((report_dir / name).is_file() for name in required):
        print(f'EN-ASR passthrough/{audio} already complete on Drive; skipping.', flush=True)
        continue
    run_streaming([sys.executable, 'scripts/benchmark_asr_v2.py', str(MANIFEST), '--direction', 'en2vi', '--split', 'test', '--audio', audio, '--denoiser', 'passthrough', '--report-dir', str(report_dir)], f'EN-ASR passthrough/{audio}')


In [ ]:
import json
{audio: json.loads((REPORT_ROOT / audio / 'aggregate.json').read_text(encoding='utf-8')) for audio in ('clean', 'noisy')}
